In [1]:
include("Peripheral_subspace_structure_improved_eig.jl")


find_minimal_projections

# Generating the Kraus Operators

## Generating the Pauli Matrices

In [2]:
function kron_identity(n)
    if n == 0
        return 1

    elseif n == 1
        return I(2)
    else
        return kron(I(2), kron_identity(n - 1))
    end
end

kron_identity (generic function with 1 method)

In [3]:
n=7 # number of qubits
X = [0 1; 1 0]
Y = [0 -im; im 0]
Z = [1 0; 0 -1]
X_exp = [cos(1) 1im*sin(1); 1im*sin(1) cos(1)]
Y_exp = [cos(1) sin(1); -sin(1) cos(1)]
Z_exp = [exp(1im) 0; 0 exp(-1im)]
X_exp_ops = [kron(kron_identity(i - 1), kron(X_exp, kron_identity(n - i))) for i in 1:n]
Y_exp_ops = [kron(kron_identity(i - 1), kron(Y_exp, kron_identity(n - i))) for i in 1:n]
Z_exp_ops = [kron(kron_identity(i - 1), kron(Z_exp, kron_identity(n - i))) for i in 1:n]

7-element Vector{Matrix{ComplexF64}}:
 [0.5403023058681398 + 0.8414709848078965im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.5403023058681398 + 0.8414709848078965im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.5403023058681398 - 0.8414709848078965im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.5403023058681398 - 0.8414709848078965im]
 [0.5403023058681398 + 0.8414709848078965im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.5403023058681398 + 0.8414709848078965im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.5403023058681398 - 0.8414709848078965im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.5403023058681398 - 0.8414709848078965im]
 [0.5403023058681398 + 0.8414709848078965im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.5403023058681398 + 0.8414709848078965im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.5403023058681398 - 0.8414709848078965im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.5403023058681

## Generate the Kraus operators for the collective noise channel

In [4]:
E_x_exp = (1 / sqrt(3)) * I(2^n)
for x in X_exp_ops
    E_x_exp = E_x_exp * x
end
E_y_exp = (1 / sqrt(3)) * I(2^n)
for y in Y_exp_ops
    E_y_exp = E_y_exp * y
end
E_z_exp = (1 / sqrt(3)) * I(2^n)
for z in Z_exp_ops
    E_z_exp = E_z_exp * z
end

In [5]:
kraus_operators = [E_x_exp, E_y_exp, E_z_exp]

3-element Vector{Matrix{ComplexF64}}:
 [0.007760632525924172 + 0.0im 0.0 + 0.012086469044082391im … -0.11074166944606728 + 0.0im 0.0 - 0.17246993143648492im; 0.0 + 0.012086469044082391im 0.007760632525924172 + 0.0im … 0.0 - 0.17246993143648492im -0.11074166944606728 + 0.0im; … ; -0.11074166944606728 + 0.0im 0.0 - 0.17246993143648492im … 0.007760632525924172 + 0.0im 0.0 + 0.012086469044082391im; 0.0 - 0.17246993143648492im -0.11074166944606728 + 0.0im … 0.0 + 0.012086469044082391im 0.007760632525924172 + 0.0im]
 [0.007760632525924172 + 0.0im 0.012086469044082391 + 0.0im … 0.11074166944606728 + 0.0im 0.17246993143648492 + 0.0im; -0.012086469044082391 + 0.0im 0.007760632525924172 + 0.0im … -0.17246993143648492 + 0.0im 0.11074166944606728 + 0.0im; … ; 0.11074166944606728 + 0.0im 0.17246993143648492 + 0.0im … 0.007760632525924172 + 0.0im 0.012086469044082391 + 0.0im; -0.17246993143648492 + 0.0im 0.11074166944606728 + 0.0im … -0.012086469044082391 + 0.0im 0.007760632525924172 + 0.0im]
 [0.43

# Start finding the structure step by step

set the variables:

In [6]:
kraus_ops= kraus_operators
threshold= 1e-6
print_structure=true
d = size(kraus_ops[1], 1)
d2 = d * d

16384

Compute the superoperator representation 

In [7]:
T = zeros(Complex{Float64}, d2, d2)

# Sum Ti ⊗ Ti* for all Kraus operators
for Ti in kraus_ops
    T += kron(Ti, conj(Ti))
end

In [8]:
using LinearAlgebra

"""
    matrix_density(T; tol=1e-14)

Compute the numerical density of matrix T.
Density = fraction of entries with magnitude > tol.
"""
function matrix_density(T; tol=1e-14)
    n, m = size(T)
    total = n * m

    nonzero_count = count(x -> abs(x) > tol, T)

    density = nonzero_count / total

    println("Matrix size: $n × $m")
    println("Total entries: $total")
    println("Entries |x| > $tol : $nonzero_count")
    println("Numerical density: $density")

    return density
end

matrix_density

In [9]:
matrix_density(T,tol=1e-6) 

Matrix size: 16384 × 16384
Total entries: 268435456
Entries |x| > 1.0e-6 : 201334784
Numerical density: 0.750030517578125


0.750030517578125

Compute the Fixed Point of the channel

In [10]:
right_eigvecs_fixed, left_eigvecs_fixed = left_and_right_eigenvectors_with_eigenvalue_one(T, threshold,nev=431)


Factorizing T - I (right side)...
Running ARPACK (right)...
Right multiplicity detected: 429
Factorizing T' - I (left side)...
Running ARPACK (left)...
Left multiplicity detected: 429


(Matrix{ComplexF64}[[-0.0007291513268271065 - 0.003058554669671898im 3.2881951130630236e-18 - 2.2087289365035936e-18im … 3.725678772135716e-18 - 2.3516717057227195e-18im -7.47547043617559e-19 - 1.7143307824417715e-18im; -1.0160723115589306e-17 - 2.600477126041035e-18im -0.0005623227756854211 - 0.003241338898902407im … -4.461629377122785e-18 - 3.721884459355461e-18im 3.84857741537453e-18 - 2.0619153805994924e-18im; … ; -3.6205169923936725e-18 + 5.5442235826679446e-18im 3.298523479313366e-18 - 2.2098423537437804e-18im … -0.0005623227756854159 - 0.0032413388989024103im 4.4301767762563546e-18 - 2.8423062597946375e-18im; -2.9596955389950023e-18 + 1.1084692468954627e-18im -2.488365756861961e-18 - 2.141603447092874e-19im … 5.453957758212716e-18 + 8.49412135749203e-19im -0.0007291513268271078 - 0.003058554669671892im], [-0.008071091508618964 + 0.002181665230588896im 2.3269645577248744e-18 - 2.5382000661498562e-18im … 2.649324744316513e-18 - 7.646671274519381e-19im 4.736613037167136e-19 - 2.473

In [11]:
 # Make  eigenvectors Hermitian and orthogonalize them
    right_eigvecs_fixed = make_hermitian_and_orthogonalize(right_eigvecs_fixed, threshold)

429-element Vector{Any}:
 ComplexF64[-0.0085407301308504 + 0.0im -4.024981154577751e-17 + 2.294321797813766e-18im … 6.158929894326942e-19 - 4.6243288819789926e-17im -2.171192540063666e-17 - 1.6532078031617218e-17im; -4.024981154577751e-17 - 2.294321797813766e-18im -0.006586625981274106 + 0.0im … -6.811873764268804e-18 - 8.855461887936924e-18im 7.966248067286943e-18 - 1.0821606248821932e-17im; … ; 6.158929894326942e-19 + 4.6243288819789926e-17im -6.811873764268804e-18 + 8.855461887936924e-18im … -0.006586625981274045 + 0.0im 5.788765824733648e-17 - 2.162100607609004e-17im; -2.171192540063666e-17 + 1.6532078031617218e-17im 7.966248067286943e-18 + 1.0821606248821932e-17im … 5.788765824733648e-17 + 2.162100607609004e-17im -0.008540730130850415 + 0.0im]
 ComplexF64[0.01753664796002857 - 8.54388683716132e-20im 1.2262694304049016e-17 + 3.951349282053642e-17im … -9.327098114030565e-18 + 1.9424354264185823e-17im 7.883750420733147e-19 + 5.730721583742716e-18im; 1.2262694304049016e-17 - 3.9513492

In [12]:
left_eigvecs_fixed = make_hermitian_and_orthogonalize(left_eigvecs_fixed, threshold)
    # Compute the projector to the support of the subspace spanned by left eigenvectors

429-element Vector{Any}:
 ComplexF64[-0.017709156835272258 + 0.0im 7.350481331322959e-18 - 2.874913784913245e-18im … 7.023021205031857e-18 - 1.0496545082181588e-17im -3.610514192644722e-18 - 5.102550398906226e-18im; 7.350481331322959e-18 + 2.874913784913245e-18im 0.005148204764233982 + 0.0im … 1.0484063047950186e-18 + 6.371291860022996e-18im -9.181593321644544e-18 + 8.652868753700641e-18im; … ; 7.023021205031857e-18 + 1.0496545082181588e-17im 1.0484063047950186e-18 - 6.371291860022996e-18im … 0.005148204764233968 + 0.0im 4.103640576278915e-18 + 9.059322320702053e-18im; -3.610514192644722e-18 + 5.102550398906226e-18im -9.181593321644544e-18 - 8.652868753700641e-18im … 4.103640576278915e-18 - 9.059322320702053e-18im -0.0177091568352722 + 0.0im]
 ComplexF64[0.03909850195909204 - 6.272451951291598e-20im 5.821145378866757e-18 + 5.1141474992534644e-18im … 3.576608820433767e-19 - 2.0448755832727744e-18im -1.2700404933150605e-17 - 1.405675218643017e-18im; 5.821145378866757e-18 - 5.114147499253

In [13]:
left_eigvecs_fixed_support_projector = projector_to_support_of_subspace(left_eigvecs_fixed, threshold)

128×128 Matrix{ComplexF64}:
          1.0+0.0im          …  -5.63785e-17-1.49186e-16im
 -4.56681e-16+1.09839e-16im      1.30035e-16+3.71297e-16im
 -3.94993e-16+8.61051e-17im      1.22449e-16+2.40495e-16im
  -1.0273e-16+1.99098e-17im     -1.79904e-16+3.27753e-16im
 -4.71241e-16+1.28512e-16im      1.55899e-16+3.39466e-16im
 -6.06699e-17-3.367e-17im    …  -4.73435e-17+2.00055e-16im
  5.07479e-17+7.79467e-17im      1.43144e-16-1.96695e-16im
  1.79728e-16-1.59975e-16im      6.87076e-17-2.55328e-16im
 -4.81232e-16+5.17981e-17im      4.15574e-17+1.96687e-16im
 -4.02594e-18+1.22295e-16im      5.23514e-17-2.27212e-19im
 -7.79841e-17+1.21457e-16im  …   2.23472e-16+2.47757e-16im
  1.95412e-16-1.54101e-16im      1.33013e-16-2.74859e-16im
  1.61015e-17+3.46669e-17im      2.88074e-17+9.29499e-17im
             ⋮               ⋱  
  5.17875e-17+1.2363e-16im        4.7076e-17-5.51496e-17im
 -1.25639e-16-1.93495e-17im     -2.79101e-16+1.2532e-16im
 -9.42694e-17-2.14286e-17im     -3.88625e-16-5.4494e-17

In [14]:
    A_basis = project_matrices(right_eigvecs_fixed, left_eigvecs_fixed_support_projector)


429-element Vector{Any}:
 ComplexF64[-0.008540730130850403 + 2.1895288505075267e-47im -3.738604395677039e-17 - 2.4494480942085634e-18im … -3.5890324693203214e-18 - 4.1668033613371594e-17im -2.0748898071670463e-17 - 1.3983759558647933e-17im; -3.738604395677042e-17 + 2.4494480942085734e-18im -0.006586625981273865 - 4.1600086798764294e-32im … 8.91061240056993e-17 - 2.9707690165744696e-16im -2.2058257488374197e-18 + 2.271953740491328e-20im; … ; -3.589032469320327e-18 + 4.166803361337163e-17im 8.910612400569934e-17 + 2.97076901657447e-16im … -0.006586625981273821 + 3.0814879110195774e-33im 4.064185286953456e-17 - 1.939197325586021e-17im; -2.074889807167047e-17 + 1.398375955864793e-17im -2.2058257488374343e-18 - 2.271953740493145e-20im … 4.0641852869534574e-17 + 1.9391973255860214e-17im -0.008540730130850422 + 6.56858655152258e-47im]
 ComplexF64[0.017536647960028576 - 8.543886837161323e-20im -6.949236387713365e-18 + 3.992684980021156e-17im … -6.110309157366584e-18 + 2.3813385971635992e-17im 

In [25]:
max_ima=0
for base in A_basis
    if maximum(abs.(real.(base))) > max_ima
        max_ima=maximum(abs.(imag.(base)))
        println("salam\t",maximum(imag.(base)),"\t",maximum(abs.(imag.(base))),"\t", max_ima)
    end
end
print(max_ima)

salam	0.9741628761237733	0.9741628761237733	0.9741628761237733
salam	0.155985787224129	0.155985787224129	0.155985787224129
salam	0.21147238486607312	0.21147238486607312	0.21147238486607312
salam	0.5150361070639471	0.5150361070639472	0.5150361070639472
salam	0.781175109090403	0.781175109090403	0.781175109090403
salam	0.8140493661052491	0.8140493661052491	0.8140493661052491
salam	0.9107188136861789	0.9107188136861789	0.9107188136861789
salam	0.11738334135279453	0.11738334135279453	0.11738334135279453
salam	0.9908739011234483	0.9908739011234483	0.9908739011234483
salam	0.35119499017425193	0.35119499017425193	0.35119499017425193
salam	0.6886736074238211	0.6886736074238211	0.6886736074238211
salam	0.5974323057054365	0.5974323057054365	0.5974323057054365
salam	0.38348733130842594	0.38348733130842594	0.38348733130842594
salam	0.9968573354998209	0.9968573354998211	0.9968573354998211
salam	0.6987201715916068	0.6987201715916068	0.6987201715916068
salam	0.9974535695215742	0.9974535695215742	0.997

## looking for the structure of algebra

In [15]:
#central_projections, minimal_projections = structure_of_algebra(A_basis, threshold)
A_matrices = A_basis;
d = size(A_matrices[1], 1);
    I_d = Matrix{Complex{Float64}}(I, d, d);  # Identity matrix of size d

In [26]:
using LinearAlgebra
using Arpack
using LinearMaps

function compute_center_of_algebra_new(
    A_matrices;
    nev::Int = 50,
    tol::Float64 = 1e-10,
    threshold::Float64 = 1e-8
)

    d = size(A_matrices[1], 1)
    d2 = d^2

    println("Building Γ as operator (no Kronecker construction)...")

    # Define Γ action on vectorized X
    function Gamma_action_vec(v)

        X = reshape(v, d, d)
        Y = zeros(ComplexF64, d, d)

        for Ai in A_matrices
            Y += Ai' * Ai * X
            Y -= Ai' * X * Ai
            Y -= Ai * X * Ai'
            Y += X * Ai * Ai'
        end

        return vec(Y)
    end

    # Create LinearMap representation
    Gamma_op = LinearMap{ComplexF64}(Gamma_action_vec, d2, d2)

    println("Computing nullspace of Γ via ARPACK...")

    # Shift-invert around 0
    # We request smallest magnitude eigenvalues
    vals, vecs = eigs(Gamma_op; nev=nev, which=:SM, tol=tol)

    idx = findall(abs.(vals) .< threshold)

    println("Nullspace dimension of Γ: ", length(idx))

    B = vecs[:, idx]

    # ----------------------------------------
    # Build Γ_p operator using B
    # ----------------------------------------

    println("Building Γ_p operator...")

    B_mats = [reshape(B[:, i], d, d) for i in 1:size(B,2)]

    function Gamma_p_action_vec(v)

        X = reshape(v, d, d)
        Y = zeros(ComplexF64, d, d)

        for Bi in B_mats
            Y += Bi' * Bi * X
            Y -= Bi' * X * Bi
            Y -= Bi * X * Bi'
            Y += X * Bi * Bi'
        end

        return vec(Y)
    end

    Gamma_total = LinearMap{ComplexF64}(
        v -> Gamma_action_vec(v) + Gamma_p_action_vec(v),
        d2, d2
    )

    println("Computing nullspace of Γ + Γ_p ...")

    vals2, vecs2 = eigs(Gamma_total; nev=nev, which=:SM, tol=tol)

    idx2 = findall(abs.(vals2) .< threshold)

    println("Center dimension: ", length(idx2))

    center_generators = [
        reshape(vecs2[:, i], d, d)
        for i in idx2
    ]

    return center_generators
end

compute_center_of_algebra_new (generic function with 1 method)

In [27]:
# Compute the center of the algebra
    A_matrices = make_hermitian_and_orthogonalize(A_matrices, threshold)
    println("let start: ")
    center_generators = compute_center_of_algebra_new(A_matrices)
    C_matrices = make_hermitian_and_orthogonalize(center_generators, threshold)

let start: 
Building Γ as operator (no Kronecker construction)...
Computing nullspace of Γ via ARPACK...
Nullspace dimension of Γ: 50
Building Γ_p operator...
Computing nullspace of Γ + Γ_p ...
Center dimension: 4


4-element Vector{Any}:
 ComplexF64[0.5038609197316453 + 0.0im 4.487114170126902e-14 + 4.350597246479531e-14im … 8.839989337134276e-14 - 5.958264576215849e-14im 6.272366866038821e-14 + 1.8496022358843982e-13im; 4.487114170126902e-14 - 4.350597246479531e-14im 0.999999999999325 + 0.0im … 9.030471480429939e-14 + 1.1419157851455895e-13im -1.06575242908493e-13 - 2.622102742280168e-13im; … ; 8.839989337134276e-14 + 5.958264576215849e-14im 9.030471480429939e-14 - 1.1419157851455895e-13im … 0.9999999999999953 + 0.0im 3.6084831613246834e-14 + 1.502774528874468e-13im; 6.272366866038821e-14 - 1.8496022358843982e-13im -1.06575242908493e-13 + 2.622102742280168e-13im … 3.6084831613246834e-14 - 1.502774528874468e-13im 0.5038609197312492 + 0.0im]
 ComplexF64[1.0 + 7.316146343364055e-32im 3.191307506044157e-14 + 5.73643371274602e-14im … 1.0718548840928305e-14 + 6.089533785359715e-14im -1.0436867745852481e-13 - 8.609316511641116e-14im; 3.191307506044157e-14 - 5.73643371274602e-14im 0.16693585092402244 + 

In [28]:
# Find the minimal projections in the center
    # ######## instead of using Identity we should use the projector to support of center
    I_d = projector_to_support_of_subspace(A_matrices, threshold)
    C_matrices = project_matrices(C_matrices, I_d) # this is for the zero part in the algebra 
    #println("trace Id=",tr(I_d))
    central_projections = find_minimal_projections(I_d, C_matrices, threshold)

    # Find the minimal projections in the algebra
    minimal_projections = []
    for P in central_projections
        projections = find_minimal_projections(P, A_matrices, threshold)
        push!(minimal_projections, projections)
    end


In [29]:
 println("Number of Sectors K= ", size(central_projections)[1])
        for i in 1:size(central_projections)[1]
            println("d_$i = ", size(minimal_projections[i])[1], "\t d'_$i = ", Int(round(real(tr(minimal_projections[i][1])), digits=1)))
        end

Number of Sectors K= 4
d_1 = 1	 d'_1 = 8
d_2 = 14	 d'_2 = 2
d_3 = 6	 d'_3 = 6
d_4 = 14	 d'_4 = 4


In [30]:
 center_generators = compute_center_of_algebra(A_matrices)
    C_matrices = make_hermitian_and_orthogonalize(center_generators, threshold)

Computing nullspace of Γ using ARPACK...
Computing nullspace of Γ + Γp ...


4-element Vector{Any}:
 ComplexF64[-1.0 + 0.0im 7.141424094879117e-13 - 1.378027006278875e-12im … 9.830210811153521e-13 - 1.551356268426771e-12im 2.5032107488606373e-16 - 1.5306694988472888e-16im; 7.141424094879117e-13 + 1.378027006278875e-12im -0.5321945554599018 + 0.0im … -4.956066706206859e-13 + 7.728493606073618e-13im -2.1636347782416166e-13 + 1.0247795863909673e-12im; … ; 9.830210811153521e-13 + 1.551356268426771e-12im -4.956066706206859e-13 - 7.728493606073618e-13im … -0.5321945554629933 + 0.0im -2.1599113356559973e-12 - 1.0737854915884432e-12im; 2.5032107488606373e-16 + 1.5306694988472888e-16im -2.1636347782416166e-13 - 1.0247795863909673e-12im … -2.1599113356559973e-12 + 1.0737854915884432e-12im -0.9999999999999867 + 0.0im]
 ComplexF64[-1.0 + 2.286447123416298e-25im 4.638057023627737e-9 + 3.0908204357828237e-9im … -7.218904044885236e-9 - 2.059855732473284e-9im -1.0063835474414107e-13 - 2.540629457715788e-13im; 4.638057023627737e-9 - 3.0908204357828237e-9im 0.33039780412073827 +

In [31]:
# Find the minimal projections in the center
    # ######## instead of using Identity we should use the projector to support of center
    I_d = projector_to_support_of_subspace(A_matrices, threshold)
    C_matrices = project_matrices(C_matrices, I_d) # this is for the zero part in the algebra 
    #println("trace Id=",tr(I_d))
    central_projections = find_minimal_projections(I_d, C_matrices, threshold)

    # Find the minimal projections in the algebra
    minimal_projections = []
    for P in central_projections
        projections = find_minimal_projections(P, A_matrices, threshold)
        push!(minimal_projections, projections)
    end
